<a href="https://colab.research.google.com/github/prjhseo/study_RS/blob/main/Bayesian_Personalized_Ranking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Implicit
- 파이썬 추천시스템 라이브러리  
https://benfred.github.io/implicit/


In [2]:
!pip install implicit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 20.0 MB/s eta 0:00:00


In [10]:
!wget https://files.grouplens.org/datasets/movielens/ml-latest-small.zip
!unzip ml-latest-small.zip

--2026-06-26 07:31:02--  https://files.grouplens.org/datasets/movielens/ml-latest-small.zip
Resolving files.grouplens.org (files.grouplens.org)... 128.101.96.204
Connecting to files.grouplens.org (files.grouplens.org)|128.101.96.204|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 978202 (955K) [application/zip]
Saving to: ‘ml-latest-small.zip’

ml-latest-small.zip 100%[===================>] 955.28K  1.12MB/s    in 0.8s    

2026-06-26 07:31:04 (1.12 MB/s) - ‘ml-latest-small.zip’ saved [978202/978202]

Archive:  ml-latest-small.zip
   creating: ml-latest-small/
  inflating: ml-latest-small/links.csv  
  inflating: ml-latest-small/tags.csv  
  inflating: ml-latest-small/ratings.csv  
  inflating: ml-latest-small/README.txt  
  inflating: ml-latest-small/movies.csv  


# 데이터 준비
- csr_matrix 형태로 데이터 불러오기

## CSR(Compressed Sparsed Row)
- 희소 행렬 표현 방법 중 하나

In [13]:
import csv
import numpy as np
from scipy.sparse import csr_matrix
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

userIds, itemIds = {}, {}
users, items = [], []

with open("ml-latest-small/ratings.csv", "r") as f:
    reader = csv.reader(f)
    print(next(reader))

    for uid,mid,_,_ in reader:
        if uid not in userIds:
            userIds[uid]=len(userIds)
        if mid not in itemIds:
            itemIds[mid]=len(itemIds)

        users.append(userIds[uid])
        items.append(itemIds[mid])

users=np.array(users)
items=np.array(items)
data=np.ones(len(users)) # 별점을 사용하지 않고 시청 여부 사용


csr=csr_matrix((data, (users, items)))# csr matrix
print(csr)


['userId', 'movieId', 'rating', 'timestamp']
<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 100836 stored elements and shape (610, 9724)>
  Coords	Values
  (0, 0)	1.0
  (0, 1)	1.0
  (0, 2)	1.0
  (0, 3)	1.0
  (0, 4)	1.0
  (0, 5)	1.0
  (0, 6)	1.0
  (0, 7)	1.0
  (0, 8)	1.0
  (0, 9)	1.0
  (0, 10)	1.0
  (0, 11)	1.0
  (0, 12)	1.0
  (0, 13)	1.0
  (0, 14)	1.0
  (0, 15)	1.0
  (0, 16)	1.0
  (0, 17)	1.0
  (0, 18)	1.0
  (0, 19)	1.0
  (0, 20)	1.0
  (0, 21)	1.0
  (0, 22)	1.0
  (0, 23)	1.0
  (0, 24)	1.0
  :	:
  (609, 9699)	1.0
  (609, 9700)	1.0
  (609, 9701)	1.0
  (609, 9702)	1.0
  (609, 9703)	1.0
  (609, 9704)	1.0
  (609, 9705)	1.0
  (609, 9706)	1.0
  (609, 9707)	1.0
  (609, 9708)	1.0
  (609, 9709)	1.0
  (609, 9710)	1.0
  (609, 9711)	1.0
  (609, 9712)	1.0
  (609, 9713)	1.0
  (609, 9714)	1.0
  (609, 9715)	1.0
  (609, 9716)	1.0
  (609, 9717)	1.0
  (609, 9718)	1.0
  (609, 9719)	1.0
  (609, 9720)	1.0
  (609, 9721)	1.0
  (609, 9722)	1.0
  (609, 9723)	1.0


# BPR in implicit

In [14]:
from implicit import bpr

# train
model=bpr.BayesianPersonalizedRanking(factors=10)
model.fit(csr)

#test
test_users=[2,3]
ids,scores=model.recommend(test_users,csr[test_users])
print(ids)
print(scores)

  0%|          | 0/100 [00:00<?, ?it/s]

[[1454 1463 3641 3014  673 1751 1449 5147  150 4742]
 [1106 3065 2677 1455 2075 1474 1109 2899 1583  864]]
[[2.420417  2.266842  2.2180936 2.2144563 2.1020205 2.0780063 2.0659113
  2.06223   2.0259507 1.9957947]
 [3.5596337 3.52512   3.501287  3.4771984 3.417025  3.3402014 3.338542
  3.3212683 3.2793093 3.2517807]]


# BPR with pytorch
선호도 : item bias + yu * yi  
평균은 1이고 user bias는 넣는게 의미가 없음(지워짐)


In [15]:
import torch

users=torch.tensor(users,device=device)
items=torch.tensor(items,device=device)

n_entries=len(items)
n_factors=10
n_items,n_users=len(itemIds),len(userIds)

item_bias=torch.zeros(n_items,requires_grad=True,device=device)
user_embs=torch.randn(n_users,n_factors,requires_grad=True,device=device)
item_embs=torch.randn(n_items,n_factors,requires_grad=True,device=device)

In [16]:
logsigmoid=torch.nn.LogSigmoid()

optim = torch.optim.Adam([item_bias, user_embs, item_embs], lr=0.01)

for epoch in range(100):
    neg_items=torch.randint(0,len(itemIds),(n_entries,),device=device) # negative sample
    pos_pref=item_bias[items]+(user_embs[users]*item_embs[items]).sum(dim=1)
    neg_pref=item_bias[neg_items]+(user_embs[users]*item_embs[neg_items]).sum(dim=1)

    cost=-logsigmoid(pos_pref-neg_pref).sum() # 내려가는 방향으로 학습하도록 -

    optim.zero_grad()
    cost.backward()
    optim.step()

    with torch.no_grad():
        if epoch%10==0:
            train_acc=sum(pos_pref>neg_pref)/n_entries
            print(f"epoch: {epoch}, train accuracy:{train_acc.item()}, cost:{cost.item()}")

epoch: 0, train accuracy:0.5015966296195984, cost:192311.53125
epoch: 10, train accuracy:0.5218275189399719, cost:163816.03125
epoch: 20, train accuracy:0.5463128089904785, cost:138924.3125
epoch: 30, train accuracy:0.5748145580291748, cost:116859.8359375
epoch: 40, train accuracy:0.6079872250556946, cost:98888.40625
epoch: 50, train accuracy:0.6430441737174988, cost:85030.2890625
epoch: 60, train accuracy:0.6815918684005737, cost:72924.0625
epoch: 70, train accuracy:0.7169761061668396, cost:63639.02734375
epoch: 80, train accuracy:0.7509421110153198, cost:55379.4296875
epoch: 90, train accuracy:0.7781248688697815, cost:49391.4140625


In [17]:
with torch.no_grad():
    n_outputs=10

    uid=2

    scores_all=item_bias+(user_embs[uid]*item_embs).sum(dim=1)
    scores_all[items[users==uid]]=0
    scores,indices=torch.topk(scores_all,n_outputs)
    print(scores)
    print(indices)

tensor([5.5666, 4.7404, 4.7274, 4.6892, 4.5645, 4.2747, 4.1550, 4.1396, 4.1213,
        4.1180])
tensor([6087, 1468, 1244, 4029, 2906,  462, 7997, 5493, 7238, 5576])
